# TOML Configuration

Save and load task configurations for hardware portability.

In [ ]:
# Check for nidaqmx
try:
    import nidaqmx
except ImportError:
    raise RuntimeError("nidaqmx not installed. Install with: pip install nidaqmx")

In [ ]:
import numpy as np
from nidaqwrapper import AITask, AOTask, DITask, DOTask
from nidaqwrapper.utils import list_devices

# Show connected hardware
devices = list_devices()
print("Connected devices:")
for idx, dev in enumerate(devices):
    print(f"  {idx}: {dev['name']} ({dev['product_type']})")

## Save AITask Config

In [ ]:
# Create AITask with channels
task = AITask("config_test", sample_rate=25600)
task.add_channel("accel_x", device="Dev1", channel_ind=0, 
                 sensitivity=100.0, sensitivity_units="mV/g", units="g")
task.add_channel("accel_y", device="Dev1", channel_ind=1,
                 sensitivity=100.0, sensitivity_units="mV/g", units="g")

# Save to TOML
task.save_config("ai_config.toml")
task.clear_task()

# Show file content
print(open("ai_config.toml").read())

## Load Config

In [ ]:
# Recreate task from TOML
loaded = AITask.from_config("ai_config.toml")
print(f"Task name: {loaded.task_name}")
print(f"Sample rate: {loaded.sample_rate} Hz")
print(f"Channels: {loaded.channel_list}")
loaded.clear_task()

## Device Remapping

To move a config to different hardware, edit the `[devices]` alias block in the
saved TOML file (each alias maps to a device name), then load it with
`from_config()`. Remapping is done by editing the file, not via a parameter.

In [ ]:
# To remap to a different chassis, edit the device names in the saved TOML.
# For example, change a line in the [devices] block from:
#     dev0 = "Dev1"
# to:
#     dev0 = "Dev2"
# then reload:
# remapped = AITask.from_config("ai_config.toml")
# print(f"Channels: {remapped.channel_list}")
# remapped.clear_task()

print("Edit the [devices] alias block in the TOML to retarget hardware.")

## AOTask Config

In [ ]:
# Create and save AOTask
ao = AOTask("sig_gen", sample_rate=10000)
ao.add_channel("ao_0", device="Dev1", channel_ind=0, min_val=-5.0, max_val=5.0)
ao.save_config("ao_config.toml")
ao.clear_task()

print(open("ao_config.toml").read())

## Digital Config

In [ ]:
# Digital input uses line specifications, not device/channel_ind
di = DITask("switches", sample_rate=1000)
di.add_channel("sw1", lines="Dev1/port0/line0:3")
di.save_config("di_config.toml")
di.clear_task()

print(open("di_config.toml").read())

## Summary

TOML configs make task definitions:
- Human-readable and editable
- Portable across hardware (edit the `[devices]` alias block to retarget)
- Version-controllable

Use `save_config()` to persist, `from_config()` to restore.